# Chroma CRUD Operations

In [1]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

#  Set Up Paths and the Vector Store

In [2]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

PosixPath('/home/rikesh/Rikesh/RAG/04_vector_stores')

In [4]:
# Load environment variables from the local .env file.
dotenv_path = "/home/rikesh/Rikesh/RAG/.env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: /home/rikesh/Rikesh/RAG/.env


In [5]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo
Persist directory: /home/rikesh/Rikesh/RAG/04_vector_stores/notebooks/chroma_langchain_db


In [6]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

No previous Chroma directory was found.


In [7]:
# Create the embedding model and connect it to a persistent Chroma store.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Vector store is ready.")

Vector store is ready.


# Add Small Helper Functions

In [8]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

# Create and Insert Example Documents

In [9]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [10]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [11]:
print(uuid4())

c5dd1133-b58b-4692-addf-c09bd8b134eb


In [12]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1. id=be899838-4904-4c17-b55f-c95d0d1f2efe
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=a47383d4-5480-4dd2-8f11-0590c0817757
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=0a0256a0-87b2-4cb2-bf52-a89b6afb30fe
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=933298d1-ae04-4873-a0cd-8b4c00bf11ee
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=cb5db4c0-e893-4c1c-85b6-ed12a4bd300b
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=f9c547d7-78a2-480b-8d61-77ddd6036ed3
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [13]:
documents[0].id

'be899838-4904-4c17-b55f-c95d0d1f2efe'

In [14]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
be899838-4904-4c17-b55f-c95d0d1f2efe
a47383d4-5480-4dd2-8f11-0590c0817757
0a0256a0-87b2-4cb2-bf52-a89b6afb30fe
933298d1-ae04-4873-a0cd-8b4c00bf11ee
cb5db4c0-e893-4c1c-85b6-ed12a4bd300b
f9c547d7-78a2-480b-8d61-77ddd6036ed3
f0ff5042-58c2-4145-ab3b-051e90f71792
7a4e6c5c-9ab2-4aeb-8374-a929db1e545d
e93dcd6e-76a3-4b71-b52c-b31fd68b59bd
9da39230-fe3c-4b84-8710-f0c9db01e744

Total inserted documents: 10


# Read the Stored Data Back

In [21]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [22]:
raw_records

{'ids': ['be899838-4904-4c17-b55f-c95d0d1f2efe',
  'a47383d4-5480-4dd2-8f11-0590c0817757',
  '0a0256a0-87b2-4cb2-bf52-a89b6afb30fe',
  '933298d1-ae04-4873-a0cd-8b4c00bf11ee',
  'cb5db4c0-e893-4c1c-85b6-ed12a4bd300b',
  'f9c547d7-78a2-480b-8d61-77ddd6036ed3',
  'f0ff5042-58c2-4145-ab3b-051e90f71792',
  '7a4e6c5c-9ab2-4aeb-8374-a929db1e545d',
  'e93dcd6e-76a3-4b71-b52c-b31fd68b59bd',
  '9da39230-fe3c-4b84-8710-f0c9db01e744'],
 'embeddings': array([[ 0.00488281,  0.02095032,  0.01655579, ...,  0.00069189,
         -0.01637268,  0.02790833],
        [-0.01448822, -0.00543976,  0.03039551, ..., -0.02853394,
         -0.00193405,  0.04272461],
        [ 0.0226593 ,  0.01534271,  0.04550171, ...,  0.02297974,
          0.00804901, -0.01585388],
        ...,
        [ 0.00803375,  0.02313232,  0.02172852, ..., -0.02070618,
         -0.01786804,  0.01334381],
        [ 0.00882721,  0.06274414,  0.09967041, ..., -0.01580811,
         -0.01374817,  0.03677368],
        [ 0.01803589,  0.08557129, 

In [17]:
print(raw_records["embeddings"][0:2, 0:20].shape)

(2, 20)


In [18]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
be899838-4904-4c17-b55f-c95d0d1f2efe
a47383d4-5480-4dd2-8f11-0590c0817757
0a0256a0-87b2-4cb2-bf52-a89b6afb30fe


In [23]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[:3]
selected_ids

['be899838-4904-4c17-b55f-c95d0d1f2efe',
 'a47383d4-5480-4dd2-8f11-0590c0817757',
 '0a0256a0-87b2-4cb2-bf52-a89b6afb30fe']

In [24]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=be899838-4904-4c17-b55f-c95d0d1f2efe
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=a47383d4-5480-4dd2-8f11-0590c0817757
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=0a0256a0-87b2-4cb2-bf52-a89b6afb30fe
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.



In [25]:
print(selected_documents)

[Document(id='be899838-4904-4c17-b55f-c95d0d1f2efe', metadata={'doc_number': 1, 'topic': 'AI'}, page_content='Artificial intelligence helps machines perform tasks that usually need human reasoning.'), Document(id='a47383d4-5480-4dd2-8f11-0590c0817757', metadata={'topic': 'AI', 'doc_number': 2}, page_content='AI systems can analyze patterns in data to support predictions and automation.'), Document(id='0a0256a0-87b2-4cb2-bf52-a89b6afb30fe', metadata={'doc_number': 3, 'topic': 'AI'}, page_content='Responsible AI development includes fairness, transparency, and safety checks.')]


# Run a Similarity Search

In [26]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [27]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=933298d1-ae04-4873-a0cd-8b4c00bf11ee
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=7a4e6c5c-9ab2-4aeb-8374-a929db1e545d
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3. id=cb5db4c0-e893-4c1c-85b6-ed12a4bd300b
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



In [28]:
search_results

[Document(id='933298d1-ae04-4873-a0cd-8b4c00bf11ee', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='7a4e6c5c-9ab2-4aeb-8374-a929db1e545d', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='cb5db4c0-e893-4c1c-85b6-ed12a4bd300b', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.')]

In [29]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='933298d1-ae04-4873-a0cd-8b4c00bf11ee', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.8067810535430908),
 (Document(id='7a4e6c5c-9ab2-4aeb-8374-a929db1e545d', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.9389221668243408),
 (Document(id='cb5db4c0-e893-4c1c-85b6-ed12a4bd300b', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  1.0750246047973633),
 (Document(id='f0ff5042-58c2-4145-ab3b-051e90f71792', metadata={'doc_number': 7, 'topic': 'LLM'}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.'),
  1.130632996559143)]

# Update Existing Documents

In [30]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['933298d1-ae04-4873-a0cd-8b4c00bf11ee',
 '7a4e6c5c-9ab2-4aeb-8374-a929db1e545d']

In [31]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=933298d1-ae04-4873-a0cd-8b4c00bf11ee
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=7a4e6c5c-9ab2-4aeb-8374-a929db1e545d
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [32]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [33]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
933298d1-ae04-4873-a0cd-8b4c00bf11ee
7a4e6c5c-9ab2-4aeb-8374-a929db1e545d


In [34]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=933298d1-ae04-4873-a0cd-8b4c00bf11ee
metadata={'topic': 'RAG', 'doc_number': 4}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=7a4e6c5c-9ab2-4aeb-8374-a929db1e545d
metadata={'topic': 'LLM', 'doc_number': 8}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [35]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [36]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=933298d1-ae04-4873-a0cd-8b4c00bf11ee
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=cb5db4c0-e893-4c1c-85b6-ed12a4bd300b
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



# Delete Documents

In [37]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['e93dcd6e-76a3-4b71-b52c-b31fd68b59bd',
 '9da39230-fe3c-4b84-8710-f0c9db01e744']

In [38]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
e93dcd6e-76a3-4b71-b52c-b31fd68b59bd
9da39230-fe3c-4b84-8710-f0c9db01e744


In [39]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
be899838-4904-4c17-b55f-c95d0d1f2efe
a47383d4-5480-4dd2-8f11-0590c0817757
0a0256a0-87b2-4cb2-bf52-a89b6afb30fe
933298d1-ae04-4873-a0cd-8b4c00bf11ee
cb5db4c0-e893-4c1c-85b6-ed12a4bd300b
f9c547d7-78a2-480b-8d61-77ddd6036ed3
f0ff5042-58c2-4145-ab3b-051e90f71792
7a4e6c5c-9ab2-4aeb-8374-a929db1e545d

Deleted ids still present?
e93dcd6e-76a3-4b71-b52c-b31fd68b59bd: False
9da39230-fe3c-4b84-8710-f0c9db01e744: False


In [40]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
